In [1]:
import torch
import pandas as pd
from linear_regression import LinearRegression
from logistic_regression import LogisticRegression
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv("../datasets/1.csv")

print("Null Values: ")
display(df.isna().sum())

print("\n\nDisplay Basic Information: ")
display(df.describe())

print("\n\nDisplay First 5 rows")
df.head()

Null Values: 


customer_id                0
age                        0
monthly_charges            0
total_charges              0
contract_length_months     0
internet_service           0
online_security            0
tech_support               0
num_complaints             0
customer_service_calls     0
account_age_months         0
payment_method             0
paperless_billing          0
service_quality_score      0
avg_response_time_hours    0
churn                      0
lifetime_value             0
subscription_score         0
dtype: int64



Display Basic Information: 


,customer_id,age,monthly_charges,total_charges,contract_length_months,num_complaints,customer_service_calls,account_age_months,service_quality_score,avg_response_time_hours,churn,lifetime_value,subscription_score
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.00000,1000.000000,1000.000000
mean,500.500000,46.247000,84.059714,4058.831426,12.237000,1.884000,4.446000,35.505000,5.406756,2.038057,0.24300,2752.285090,99.787508
std,288.819436,16.288072,37.688858,2281.651747,9.002548,1.430591,2.926029,20.366757,2.588228,1.929573,0.42911,2156.526099,53.824790
min,1.000000,18.000000,20.602163,112.364331,1.000000,0.000000,0.000000,1.000000,1.009783,0.002631,0.00000,0.000000,0.000000
25%,250.750000,33.000000,50.144172,2147.109445,1.000000,1.000000,2.000000,17.000000,3.281492,0.614179,0.00000,1028.046730,59.626709
50%,500.500000,46.000000,84.391460,4097.378495,12.000000,2.000000,4.000000,36.000000,5.337851,1.460773,0.00000,2128.452440,98.745100
75%,750.250000,60.250000,115.877796,6053.135199,24.000000,3.000000,7.000000,53.000000,7.581240,2.923837,0.00000,4076.180036,137.948784
max,1000.000000,74.000000,149.923784,7986.945340,24.000000,4.000000,9.000000,71.000000,9.996721,14.005820,1.00000,10667.912066,200.000000




Display First 5 rows


,customer_id,age,monthly_charges,total_charges,contract_length_months,internet_service,online_security,tech_support,num_complaints,customer_service_calls,account_age_months,payment_method,paperless_billing,service_quality_score,avg_response_time_hours,churn,lifetime_value,subscription_score
0,1,56,84.040352,4796.363048,12,Fiber optic,No,No,3,9,41,Bank transfer,No,5.043339,0.273458,0,3272.965520,76.251558
1,2,69,62.737709,4024.458185,12,No,No,No,1,7,47,Electronic check,Yes,6.875320,5.910615,1,2103.946610,132.891972
2,3,46,102.342111,7903.505610,12,DSL,No,No,1,9,54,Electronic check,Yes,8.705315,1.696716,0,5694.437607,182.184774
3,4,32,51.218930,1177.874045,12,Fiber optic,No,Yes,0,8,55,Check,Yes,3.584187,0.729308,1,2032.124365,66.584825
4,5,60,29.862233,5591.641977,1,DSL,No,No,2,0,19,Check,No,2.653555,1.368472,1,686.439036,92.387701


In [3]:
internet_service = pd.get_dummies(
    df["internet_service"],
    prefix="internet_service"
)

# Rename columns
internet_service.columns = [
    col.lower().replace(" ", "_") for col in internet_service.columns
]

internet_service = internet_service.astype(int)
internet_service.head()

,internet_service_dsl,internet_service_fiber_optic,internet_service_no
0,0,1,0
1,0,0,1
2,1,0,0
3,0,1,0
4,1,0,0


In [4]:
df = pd.concat([df, internet_service], axis=1)
df["online_security"] = df["online_security"].map({"Yes":1, "No":0})
df["tech_support"] = df["tech_support"].map({"Yes":1, "No":0})


df = df.drop(columns=["internet_service", "payment_method", "paperless_billing", "customer_id"])

df.head()

,age,monthly_charges,total_charges,contract_length_months,online_security,tech_support,num_complaints,customer_service_calls,account_age_months,service_quality_score,avg_response_time_hours,churn,lifetime_value,subscription_score,internet_service_dsl,internet_service_fiber_optic,internet_service_no
0,56,84.040352,4796.363048,12,0,0,3,9,41,5.043339,0.273458,0,3272.965520,76.251558,0,1,0
1,69,62.737709,4024.458185,12,0,0,1,7,47,6.875320,5.910615,1,2103.946610,132.891972,0,0,1
2,46,102.342111,7903.505610,12,0,0,1,9,54,8.705315,1.696716,0,5694.437607,182.184774,1,0,0
3,32,51.218930,1177.874045,12,0,1,0,8,55,3.584187,0.729308,1,2032.124365,66.584825,0,1,0
4,60,29.862233,5591.641977,1,0,0,2,0,19,2.653555,1.368472,1,686.439036,92.387701,1,0,0


In [5]:
import numpy as np

def normalize(df, skips=[]):
    skip = df[skips]
    df = df.drop(columns=skips)
    
    remaining_cols = df.columns.tolist()
    for cols in remaining_cols:
        np_arr = df[cols].to_numpy()
        np_arr = (np_arr - np.nanmin(np_arr))/(np.nanmax(np_arr) - np.nanmin(np_arr))
        df[cols] = np_arr
    df = pd.concat([df, skip], axis=1)
    return df
        

In [6]:
df = normalize(df, skips=["online_security", "tech_support", "churn", "internet_service_dsl", "internet_service_fiber_optic", "internet_service_no"])
df.head()

,age,monthly_charges,total_charges,contract_length_months,num_complaints,customer_service_calls,account_age_months,service_quality_score,avg_response_time_hours,lifetime_value,subscription_score,online_security,tech_support,churn,internet_service_dsl,internet_service_fiber_optic,internet_service_no
0,0.678571,0.490546,0.594825,0.478261,0.75,1.000000,0.571429,0.448824,0.019340,0.306805,0.381258,0,0,0,0,1,0
1,0.910714,0.325820,0.496800,0.478261,0.25,0.777778,0.657143,0.652674,0.421903,0.197222,0.664460,0,0,1,0,0,1
2,0.500000,0.632067,0.989404,0.478261,0.25,1.000000,0.757143,0.856302,0.120978,0.533791,0.910924,0,0,0,1,0,0
3,0.250000,0.236749,0.135310,0.478261,0.00,0.888889,0.771429,0.286461,0.051894,0.190489,0.332924,0,1,1,0,1,0
4,0.750000,0.071605,0.695818,0.000000,0.50,0.000000,0.257143,0.182907,0.097538,0.064346,0.461939,0,0,1,1,0,0


<h1>Linear Regression</h1>

In [7]:
Y = df["lifetime_value"]
X = df.drop(columns=["lifetime_value"])

X_train, X_test, y_train, y_test = train_test_split(X.values, Y.values, test_size=0.2, random_state=42)

In [8]:
linear_model = LinearRegression(X_train.shape[1])

In [9]:
for epoch in range(3200):
    loss = linear_model.fit(X_train, y_train, 32, 0.001)
    if (epoch+1)%200==0:
        print(f"EPOCH: {epoch+1} =====================> LOSS: {loss}")

EPOCH: 200 =====================> LOSS: 0.04036934666335583
EPOCH: 400 =====================> LOSS: 0.0159678740799427
EPOCH: 600 =====================> LOSS: 0.009731260910630226
EPOCH: 800 =====================> LOSS: 0.00755700271576643
EPOCH: 1000 =====================> LOSS: 0.0065818150900304314
EPOCH: 1200 =====================> LOSS: 0.006068549612537027
EPOCH: 1400 =====================> LOSS: 0.005772437872365117
EPOCH: 1600 =====================> LOSS: 0.0055920935329049825
EPOCH: 1800 =====================> LOSS: 0.005478465259075165
EPOCH: 2000 =====================> LOSS: 0.0054052611719816925
EPOCH: 2200 =====================> LOSS: 0.005357381645590067
EPOCH: 2400 =====================> LOSS: 0.005325740985572338
EPOCH: 2600 =====================> LOSS: 0.005304677877575159
EPOCH: 2800 =====================> LOSS: 0.005290574925020337
EPOCH: 3000 =====================> LOSS: 0.005281095849350095
EPOCH: 3200 =====================> LOSS: 0.0052747017983347175


In [10]:
from sklearn.metrics import r2_score

y_pred = linear_model.predict(X_test)

y_pred = y_pred.detach().cpu().numpy()
y_test = y_test if isinstance(y_test, np.ndarray) else np.array(y_test)

r2 = r2_score(y_test, y_pred)
print(f"R2 Score => {r2}")

R2 Score => 0.878606203601497


<h1>Logistic Regression</h1>

In [11]:
Y = df["churn"]
X = df.drop(columns=["churn"])

X_train, X_test, y_train, y_test = train_test_split(X.values, Y.values, test_size=0.2, random_state=42)

In [12]:
logistic_model = LogisticRegression(X_train.shape[1])

In [13]:
for epoch in range(3200):
    loss = logistic_model.fit(X_train, y_train, 32, 0.001)
    if (epoch+1)%200==0:
        print(f"EPOCH: {epoch+1} =====================> LOSS: {loss}")

NameError: name 'log' is not defined

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

y_pred = logistic_model.predict(X_test)

# GPU → CPU
y_pred = y_pred.detach().cpu().numpy()
y_test = np.array(y_test)

# probability → class
y_pred_class = (y_pred > 0.5).astype(int)

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("Accuracy:", accuracy_score(y_test, y_pred_class))
print("Precision:", precision_score(y_test, y_pred_class))
print("Recall:", recall_score(y_test, y_pred_class))
print("F1:", f1_score(y_test, y_pred_class))